# Setting up Milvus Cache

In [2]:
from pymilvus import connections
from pymilvus import db, Collection
from pymilvus import utility

conn_name = "cache_conn"
db_name = "cache_db"
collection_name = "llm_cache"

connections.add_connection(
    cache_conn = {
        "host": "localhost",
        "port": "19530",
        "username": "username",
        "password": "password",
    }
)

#Connect
connections.connect(conn_name)

#Create a DB if not already present
current_dbs = db.list_database(using=conn_name)

if (db_name not in current_dbs):
    print("Creating DB: ", db_name)
    resume_db = db.create_database(db_name, using=conn_name)
else:
    print(db_name, " DB already exists")

#Switch to the new db
db.using_database(db_name, using=conn_name)

Creating DB:  cache_db


In [5]:
# Creating Collection
from pymilvus import CollectionSchema, FieldSchema, DataType, Collection
import json

cache_id = FieldSchema(
    name="cache_id",
    dtype=DataType.INT64,
    auto_id=True,
    is_primary=True,
    max_length=32
)
prompt_text = FieldSchema(
    name="prompt_text",
    dtype=DataType.VARCHAR,
    max_length=2048
)
response_text = FieldSchema(
    name="response_text",
    dtype=DataType.VARCHAR,
    max_length=2048
)
prompt_embedding = FieldSchema(
    name="prompt_embedding",
    dtype=DataType.FLOAT_VECTOR,
    dim=1536
)

# Creating Schema
cache_schema = CollectionSchema(
    fields=[cache_id, prompt_text, response_text, prompt_embedding],
    description="Cache for LLM",
    enable_dynamic_field=True
)

#Create Collection
cache_collection = Collection(
    name=collection_name,
    schema=cache_schema,
    using=conn_name,
    shard_num=2
)
print("Schema: ", cache_collection.schema, "\n")

#Indexing
index_params = {
    "metric_type": "L2",
    "index_type": "IVF_FLAT",
    "params": { "nlist": 1024 }
}

cache_collection.create_index(
    field_name="prompt_embedding",
    index_params=index_params
)
cache_collection.flush()
cache_collection.load()

Schema:  {'auto_id': True, 'description': 'Cache for LLM', 'fields': [{'name': 'cache_id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': True}, {'name': 'prompt_text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 2048}}, {'name': 'response_text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 2048}}, {'name': 'prompt_embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 1536}}], 'enable_dynamic_field': True} 



# Inference Process with Caching

In [14]:
from transformers import AutoTokenizer
from langchain.llms import OpenAI
from langchain.embeddings import OpenAIEmbeddings
import os
import time
#Setup open API key to use OpenAI's LLM
openai_api_key="abc123"
os.environ["ΟΡΕΝΑΙ_ΑΡΙ_ΚΕΥ"] = openai_api_key
#Create an LLM object
llm = OpenAI(temperature=0, model="gpt-3.5-turbo-instruct")
#Setup embedding model for creating embeddings
embeddings_model = OpenAIEmbeddings()

#setup threshold for similarity between vectors
similarity_threshold = 0.3
search_params = {
    "metric_type": "L2",
    "offset": 0,
    "ignore_growing": False,
    "params": { "nprobe": 20, "radius": similarity_threshold }
}
#create a function to run the inference loop
def get_response(prompt):
    start_time=time.time()
    #create embedding for incoming prompt
    prompt_embed=embeddings_model.embed_query(prompt)

    #Check cache if result exists.
    cache_results=cache_collection.search(
        data=[prompt_embed],
        anns_field="prompt_embedding",
        param=search_params,
        limit=1, #Look for the top result only
        expr=None,
        output_fields=["prompt_text", "response_text"],
        consistency_level="Strong"
    )
    returned_response ="None"

    if (len(cache_results[0]) > 0):
        #Cache hit
        print(prompt, ":\n Cache hit: ", cache_results[0])
        returned_response = cache_results[0][0].entity.get("response_text") 
    else:
        #Find answer with LLM
        llm_response = llm(prompt)
        print(prompt, "\n LLM returned", llm_response)
        returned_response = llm_response
        #save prompt/response to cache
        prompt_text = [prompt]
        prompt_embedding=[prompt_embed]
        response_text = [llm_response]
        insert_data=[prompt_text, response_text, prompt_embedding]
        mr=cache_collection.insert(insert_data)
        
    end_time = time.time()
    print("Time elapsed: ", end_time - start_time, "\n")
    return returned_response


In [16]:
#Build up the cache
response=get_response("In which year was Abraham Lincoln born?")
response=get_response("What is distance between the sun and the moon?")
response=get_response("How many years have Lebron James played in the NBA?")
response=get_response("What are the advantages of the python language?")
response=get_response("What is the typical height of an elephant")

In which year was Abraham Lincoln born? :
 Cache hit:  ["id: 464414954947367310, distance: 0.0, entity: {'prompt_text': 'In which year was Abraham Lincoln born?', 'response_text': '\\n\\nAbraham Lincoln was born on February 12, 1809.'}"]
Time elapsed:  0.6231417655944824 

What is distance between the sun and the moon? :
 Cache hit:  ["id: 464414954947367312, distance: 0.0, entity: {'prompt_text': 'What is distance between the sun and the moon?', 'response_text': '\\n\\nThe average distance between the sun and the moon is about 93 million miles (149.6 million kilometers). However, the distance can vary due to the elliptical orbits of both bodies. At its closest point (perihelion), the distance between the sun and the moon can be about 91.4 million miles (147.1 million kilometers), while at its farthest point (aphelion), the distance can be about 94.5 million miles (152.1 million kilometers).'}"]
Time elapsed:  0.6494960784912109 

How many years have Lebron James played in the NBA? 
 L

In [17]:
response=get_response("List some advantages of the python language")
response=get_response("How tall is an elephant?")

List some advantages of the python language :
 Cache hit:  ["id: 464414954947367316, distance: 0.04901907593011856, entity: {'response_text': '\\n\\n1. Easy to Learn and Use: Python has a simple and easy-to-understand syntax, making it a popular choice for beginners. Its code is highly readable and requires fewer lines of code compared to other programming languages, making it easier to maintain and debug.\\n\\n2. Versatile and Flexible: Python can be used for a wide range of applications, including web development, data analysis, artificial intelligence, machine learning, and scientific computing. It also supports multiple programming paradigms, such as object-oriented, functional, and procedural programming.\\n\\n3. Large Standard Library: Python has a vast standard library that provides a wide range of built-in functions and modules, making it easier to develop complex applications without having to write code from scratch.\\n\\n4. Third-Party Libraries and Frameworks: Python has a 